In [1]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

fx = spark.read.table("lh_bronze.dbo.ecb_fx_usd_eur")
fx.show(10)

calendar = spark.sql("""
    SELECT explode(
        sequence(
            (SELECT MIN(rate_date) FROM lh_bronze.dbo.ecb_fx_usd_eur),
            current_date(),                                   -- (A) end at today
            interval 1 day
        )
    ) AS rate_date
""")

df_res = calendar.join(fx, "rate_date", "left")

df_res = df_res.withColumn("is_filled", F.col("usd_per_eur").isNull())

w = Window.orderBy("rate_date").rowsBetween(Window.unboundedPreceding, 0)

df_res = df_res.withColumn(
    "usd_per_eur",
    F.last("usd_per_eur", ignorenulls=True).over(w)
)

df_fx = df_res.select("rate_date", "usd_per_eur", "is_filled")

df_fx.show(10)

nulls = df_fx.filter(F.col("usd_per_eur").isNull()).count()
if nulls > 0:
    raise ValueError(f"FX table still has {nulls} days without a rate")

# (B) Summary for your documentation
df_fx.select(
    F.count("*").alias("rows"),
    F.sum(F.col("is_filled").cast("int")).alias("filled_days")
).show()

df_fx.write.mode("overwrite").format("delta").saveAsTable("lh_silver.dbo.fx_usd_eur_daily")

StatementMeta(, e1129508-274d-4a79-9f90-4ad673ae0102, 3, Finished, Available, Finished, False)

+--------+--------------+----------+-----------+
|currency|currency_denom| rate_date|usd_per_eur|
+--------+--------------+----------+-----------+
|     USD|           EUR|2015-01-02|     1.2043|
|     USD|           EUR|2015-01-05|     1.1915|
|     USD|           EUR|2015-01-06|     1.1914|
|     USD|           EUR|2015-01-07|     1.1831|
|     USD|           EUR|2015-01-08|     1.1768|
|     USD|           EUR|2015-01-09|     1.1813|
|     USD|           EUR|2015-01-12|     1.1804|
|     USD|           EUR|2015-01-13|     1.1782|
|     USD|           EUR|2015-01-14|     1.1775|
|     USD|           EUR|2015-01-15|     1.1708|
+--------+--------------+----------+-----------+
only showing top 10 rows
+----------+-----------+---------+
| rate_date|usd_per_eur|is_filled|
+----------+-----------+---------+
|2015-01-02|     1.2043|    false|
|2015-01-03|     1.2043|     true|
|2015-01-04|     1.2043|     true|
|2015-01-05|     1.1915|    false|
|2015-01-06|     1.1914|    false|
|2015-01-

In [2]:
gd = spark.read.table("lh_bronze.dbo.worldbank_gdp")
gd = (
    gd
    .select("country_code", "year", "gdp_usd")
    .filter(F.col("gdp_usd").isNotNull())
    .withColumn("gdp_usd_trillion", F.col("gdp_usd") / 1_000_000_000_000)
)

gd.show(10)
print(gd.count())

gd.write.mode("overwrite").format("delta").saveAsTable("lh_silver.dbo.gdp_yearly")

StatementMeta(, e1129508-274d-4a79-9f90-4ad673ae0102, 4, Finished, Available, Finished, False)

+------------+----+------------+----------------+
|country_code|year|     gdp_usd|gdp_usd_trillion|
+------------+----+------------+----------------+
|         USA|2025|  3.07697E13|         30.7697|
|         USA|2024|2.9298013E13|       29.298013|
|         USA|2023|2.7811517E13|       27.811517|
|         USA|2022|2.6054614E13|       26.054614|
|         USA|2021|2.3725645E13|       23.725645|
|         USA|2020|2.1375281E13|       21.375281|
|         USA|2019|2.1539982E13|       21.539982|
|         USA|2018|2.0656516E13|       20.656516|
|         USA|2017|1.9612102E13|       19.612102|
|         USA|2016|1.8804913E13|       18.804913|
+------------+----+------------+----------------+
only showing top 10 rows
26
